## Rag application using type sense

## Step 1: Import the Typesense client

In [ ]:
# Step 1: Import the Typesense Python client
import typesense

## Step 2: Connect to Typesense and create the books collection

In [ ]:
# Step 2: Connect to Typesense Cloud (host and API key are read from .env) and create the books collection
import os
from dotenv import load_dotenv

load_dotenv()

client = typesense.Client({
    'nodes': [{
        'host': os.environ['TYPESENSE_HOST'],  # Typesense Cloud host: xxx.a1.typesense.net
        'port': '443',
        'protocol': 'https'  # Typesense Cloud requires https on port 443
    }],
    'api_key': os.environ['TYPESENSE_API_KEY'],
    'connection_timeout_seconds': 2
})

books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'author', 'type': 'string', 'facet': True},
        {'name': 'publication_year', 'type': 'int32', 'facet': True},
        {'name': 'average_rating', 'type': 'float', 'facet': True},
        {'name': 'img_url', 'type': 'string', 'index': False, 'optional': True},
        {'name': 'rating_count', 'type': 'int32'},
    ],
    'default_sorting_field': 'rating_count'
}


# Create the collection only if it doesn't exist yet, so this cell can be re-run
try:
    print(client.collections['books'].retrieve())
except typesense.exceptions.ObjectNotFound:
    print(client.collections.create(books_schema))

In [ ]:
# Show the client object to confirm the connection was set up
client

## Step 3: Import books.jsonl into the books collection

In [ ]:
# Step 3: Read books.jsonl (one JSON book per line) and import every book into the books collection
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()

# 'upsert' lets you re-run this cell without "document already exists" errors
results = client.collections['books'].documents.import_(data.encode('utf-8'), {'action': 'upsert'})

## Step 4: Keyword search with filter_by

In [ ]:
# Step 4: Search titles for 'harry potter', keep only books published before 1998, newest first
search_parameters = {
    'q'         : 'harry potter',
    'query_by'  : 'title',
    'filter_by' : 'publication_year:<1998',
    'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

## Step 5: Keyword search with facet_by

In [ ]:
# Step 5: Same search, but facet_by counts the matching books per author instead of removing any, highest rated first
search_parameters = {
    'q'         : 'harry potter',
    'query_by'  : 'title',
    'facet_by'  : 'author',
    'sort_by'   : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

### Langchain + Typesense + Groq LLM + RAG Application

## Step 6: Import LangChain components

In [ ]:
# Step 6: Import the loader, splitter, embeddings, Typesense vector store and Groq chat model
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings  # langchain 1.x removed langchain.embeddings.HuggingFaceEmbeddings
from langchain_groq import ChatGroq

## Step 7: Load the Groq API key

In [ ]:
# Step 7: Read the Groq API key from .env (used to create the ChatGroq LLM)
groq_api_key = os.getenv("GROQ_API_KEY")

## Step 8: Load, split and embed test.txt

In [ ]:
# Step 8: Load test.txt, split it into ~1000-character chunks with 100 characters of overlap and set up the embedding model
# (HuggingFaceEmbeddings downloads sentence-transformers/all-mpnet-base-v2 on the first run)
loader = TextLoader("test.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

## Delete the old lang-chain collection (avoids duplicate chunks)

In [ ]:
# Delete the 'lang-chain' collection if it already exists, so re-running Step 9 stores the chunks only once
collection_name = "lang-chain"

existing_collections = [c["name"] for c in client.collections.retrieve()]

if collection_name in existing_collections:
    client.collections[collection_name].delete()
    print(f"Deleted old collection: {collection_name}")
else:
    print(f"Collection '{collection_name}' does not exist yet, nothing to delete")


## Step 9: Store the chunks in Typesense

In [ ]:
# Step 9: Embed every chunk and store it in the 'lang-chain' Typesense collection
# (re-running this adds the chunks again, creating duplicates)
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": os.environ["TYPESENSE_HOST"],
        "port": "443",
        "protocol": "https",
        "typesense_api_key": os.environ["TYPESENSE_API_KEY"],
        "typesense_collection_name": "lang-chain",
    },
)


## Step 10: Similarity search

In [ ]:
# Step 10: Find the chunks closest in meaning to the query and print the best match
query = "What is the artificial intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content )

## Step 11: Create a retriever

In [ ]:
# Step 11: Wrap the vector store as a retriever so it can plug into a LangChain RAG chain
retriever = docsearch.as_retriever()
retriever

## Step 12: Retrieve chunks for a query

In [ ]:
# Step 12: Fetch the most relevant chunks for the query and show the top one
query = "artificial intelligence indepth explanation"
retriever.invoke(query)[0]

## Step 13: Set up the Groq LLM

In [ ]:
# Step 13: Set up the Groq LLM (groq_api_key was loaded in Step 7)
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)


## Step 14: RAG function (retrieve -> context -> answer)

In [ ]:
# Step 14: RAG function - retrieve chunks from Typesense, build a prompt with that context, and ask the LLM
from langchain_core.prompts import PromptTemplate


def rag_typesense(
    query: str,
    retriever,
    llm: ChatGroq,
    top_k: int = 3
):
    """
    Simple RAG on Typesense: retrieve relevant chunks, put them in the prompt,
    and let the Groq LLM answer using only that context.

    Returns:
        The answer string
    """

    results = retriever.invoke(query)[:top_k]
    context = "

".join(doc.page_content for doc in results)

    if not context:
        return "No relevant context found."

    prompt = PromptTemplate(
        template="""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:""",
        input_variables=["context", "query"]
    )

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


## Run the RAG pipeline

In [ ]:
# Ask a question that is in test.txt, then one that is not, to check the "I don't know" behaviour
print(rag_typesense("What is artificial intelligence?", retriever, llm))
print()
print(rag_typesense("Who won the 2018 football world cup?", retriever, llm))
